In [15]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

TOROOTPATH   = "../../../../model/"
ORIGINALPATH = TOROOTPATH + "data/raw/Microsoft_Azure_Predictive_Maintenance/microsoft_azure_mix_col.csv"
OUT_DIR      = TOROOTPATH + "data/process/Microsoft Azure/"
X_TRAIN = OUT_DIR+"train/X_TRAIN.csv"
X_TEST = OUT_DIR+"test/X_TEST.csv"
Y_TRAIN = OUT_DIR+"train/Y_TRAIN.csv"
Y_TEST = OUT_DIR+"test/Y_TEST.csv"

TARGET = "days_to_failure"
TEST_SIZE = 0.2
RANDOM_STATE = 42

# columns ที่ "รู้อนาคต" ห้ามใช้เป็น feature เด็ดขาด (target leakage)
LEAKAGE_COLS = ["failures", "has_failure", "next_failure_date"]

# columns ที่เป็น raw event string ไม่เอาเข้า model ตรง ๆ (เก็บ flag แทน)
DROP_RAW = ["errors", "maint_comp", "datetime", "machineID"]

In [16]:
def load_data(path):
    df = pd.read_csv(path, parse_dates=["datetime", "next_failure_date"])
    print(f"[load] {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df
 
 
def drop_missing_target(df):
    """ตัด row ที่ไม่มี days_to_failure (อยู่หลัง failure สุดท้ายของเครื่อง)"""
    before = len(df)
    df = df.dropna(subset=[TARGET]).reset_index(drop=True)
    print(f"[target] drop {before - len(df):,} rows ที่ target เป็น NaN -> เหลือ {len(df):,}")
    return df
 
 
def feature_engineering(df):
    """สร้าง feature เพิ่มจาก sensor + event แบบ per-machine time-aware"""
    df = df.sort_values(["machineID", "datetime"]).reset_index(drop=True)
 
    # 1) เวลาจากปฏิทิน
    df["hour"]      = df["datetime"].dt.hour
    df["dayofweek"] = df["datetime"].dt.dayofweek
 
    # 2) rolling stats ของ sensor (หน้าต่าง 24 ชม.) ต่อเครื่อง
    #    degradation เป็น trend การดูแค่ snapshot เดียวไม่พอ
    sensors = ["volt", "rotate", "pressure", "vibration"]
    g = df.groupby("machineID")
    for s in sensors:
        df[f"{s}_roll_mean_24h"] = g[s].transform(lambda x: x.rolling(24, min_periods=1).mean())
        df[f"{s}_roll_std_24h"]  = g[s].transform(lambda x: x.rolling(24, min_periods=1).std().fillna(0))
 
    # 3) error สะสมต่อเครื่อง (ยิ่งสะสมมาก ยิ่งเสี่ยง)
    df["cum_errors"] = g["has_error"].cumsum()
    df["cum_maint"]  = g["has_maint"].cumsum()
 
    print(f"[feature] เพิ่ม feature -> {df.shape[1]} cols")
    return df
 
 
def encode_categorical(df):
    """one-hot encode model (model1-4)"""
    df = pd.get_dummies(df, columns=["model"], prefix="model", dtype=int)
    print(f"[encode] one-hot model -> {df.shape[1]} cols")
    return df
 
 
def build_feature_matrix(df):
    """แยก X, y และตัด leakage + raw cols ออก"""
    y = df[TARGET].copy()
    drop_cols = LEAKAGE_COLS + DROP_RAW + [TARGET]
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    # safety: เหลือแต่ตัวเลข
    X = X.select_dtypes(include=[np.number])
    print(f"[matrix] X = {X.shape}, y = {y.shape}")
    print(f"[matrix] features: {list(X.columns)}")
    return X, y
 
 
def split_and_save(X, y):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    for p in [X_TRAIN, X_TEST, Y_TRAIN, Y_TEST]:
        os.makedirs(os.path.dirname(p), exist_ok=True)
    X_tr.to_csv(X_TRAIN, index=False)
    X_te.to_csv(X_TEST, index=False)
    y_tr.to_csv(Y_TRAIN, index=False)
    y_te.to_csv(Y_TEST, index=False)
    print(f"[split] train={len(X_tr):,}  test={len(X_te):,}")
    print(f"[save]  {X_TRAIN}\n        {X_TEST}\n        {Y_TRAIN}\n        {Y_TEST}")

In [17]:
df = load_data(ORIGINALPATH)
df = drop_missing_target(df)
df = feature_engineering(df)
df = encode_categorical(df)
X, y = build_feature_matrix(df)
split_and_save(X, y)
print("\n[done] clean เสร็จสมบูรณ์")

[load] 876,100 rows x 16 cols
[target] drop 120,746 rows ที่ target เป็น NaN -> เหลือ 755,354
[feature] เพิ่ม feature -> 28 cols
[encode] one-hot model -> 31 cols
[matrix] X = (755354, 23), y = (755354,)
[matrix] features: ['age', 'volt', 'rotate', 'pressure', 'vibration', 'has_error', 'has_maint', 'hour', 'dayofweek', 'volt_roll_mean_24h', 'volt_roll_std_24h', 'rotate_roll_mean_24h', 'rotate_roll_std_24h', 'pressure_roll_mean_24h', 'pressure_roll_std_24h', 'vibration_roll_mean_24h', 'vibration_roll_std_24h', 'cum_errors', 'cum_maint', 'model_model1', 'model_model2', 'model_model3', 'model_model4']
[split] train=604,283  test=151,071
[save]  ../../../../model/data/process/Microsoft Azure/train/X_TRAIN.csv
        ../../../../model/data/process/Microsoft Azure/test/X_TEST.csv
        ../../../../model/data/process/Microsoft Azure/train/Y_TRAIN.csv
        ../../../../model/data/process/Microsoft Azure/test/Y_TEST.csv

[done] clean เสร็จสมบูรณ์
